# Generator Danych - Obróbka Mechaniczna

1. Konfiguracja

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import weibull_min

# Ustawienie ziarna w celu uzyskania tych samych wyników co uruchomienie
np.random.seed(42)

2. Definicja funkcji generującej dane

In [ ]:
def generate_machining_dataset(n_samples=30000):
    print("Rozpoczęto generowanie zbioru danych...")

    # PARAMETRY WEJŚCIOWE I MATERIAŁOWE
    # Proces frezowania aluminium (np. AlSi9Cu3)
    vc = np.random.normal(400, 15, n_samples)       # Prędkość skrawania [m/min]
    f = np.random.normal(0.09, 0.008, n_samples)    # Posuw [mm/obr]
    ap = np.random.normal(2.0, 0.1, n_samples)      # Głębokość [mm]
    hb = np.random.normal(90, 4, n_samples)         # Twardość odlewu [Brinell]

    # Inicjalizacja tablic dla zmiennych zależnych od cyklu życia narzędzia
    vb_wear = np.zeros(n_samples)
    tool_id = np.zeros(n_samples, dtype=int)

    # SYMULACJA ZUŻYCIA NARZĘDZIA (Cykl życia modelowany rozkładem Weibulla)
    current_idx = 0
    current_tool = 1

    while current_idx < n_samples:
        # Losowanie żywotności narzędzia (w sztukach detali)
        # Parametr kształtu c=2.5 (charakterystyczny dla zużycia zmęczeniowego)
        # Średnio narzędzie wytrzymuje około 800 sztuk
        life_span = int(weibull_min.rvs(c=2.5, scale=800))

        # Zapobieganie wyjściu poza rozmiar tablicy
        end_idx = min(current_idx + life_span, n_samples)
        actual_life = end_idx - current_idx

        # Zużycie VB rośnie nieliniowo (szybko na początku, potem stabilnie, na końcu lawinowo)
        time_steps = np.linspace(0, 1, actual_life)
        # Model zużycia: f(t) = a*t + b*t^3 + szum
        wear_curve = 0.15 * time_steps + 0.15 * (time_steps**3) + np.random.normal(0, 0.005, actual_life)
        wear_curve = np.clip(wear_curve, 0.0, 0.4) # Maksymalne zużycie to 0.4 mm

        vb_wear[current_idx:end_idx] = wear_curve
        tool_id[current_idx:end_idx] = current_tool

        current_idx = end_idx
        current_tool += 1

    # ZMIENNE PROCESOWE (Oparte na prawach fizyki skrawania)
    # Moc wrzeciona zależna od objętości usuwanego materiału (MRR) i stępienia narzędzia
    mrr = vc * f * ap
    power_kw = 0.02 * mrr * (1 + 0.01 * (hb - 90)) * (1 + 1.5 * vb_wear) + np.random.normal(0, 0.2, n_samples)

    # Temperatura narzędzia (wpływ tarcia i zużycia VB)
    temp_tool = 25 + 0.3 * vc + 200 * f + 50 * ap + 300 * vb_wear + np.random.normal(0, 5, n_samples)

    # Wibracje (rosną lawinowo przy mocnym zużyciu narzędzia)
    vibrations = 0.5 + 5 * f + 0.2 * ap + 15 * (vb_wear**2.5) + np.random.normal(0, 0.1, n_samples)

    # Ciśnienie chłodziwa (stabilne z drobnym szumem)
    coolant_press = np.random.normal(20.0, 0.5, n_samples)

    # WPROWADZENIE ANOMALII
    # A) Pęknięcie narzędzia (Nagły, rzadki pik - 0.2% szans, najczęściej gdy narzędzie jest zużyte)
    break_prob = np.where(vb_wear > 0.25, 0.005, 0.0001)
    tool_break_flag = np.random.binomial(1, break_prob)

    vibrations = np.where(tool_break_flag == 1, vibrations * 5 + 10, vibrations)
    power_kw = np.where(tool_break_flag == 1, power_kw * 1.5, power_kw)

    # B) Błąd czujnika temperatury chłodziwa (0.5% szans na zgubienie pakietu danych -> -99.0)
    coolant_temp = 22.0 + np.random.normal(0, 1.0, n_samples)
    sensor_error_flag = np.random.binomial(1, 0.005, n_samples)
    coolant_temp = np.where(sensor_error_flag == 1, -99.0, coolant_temp)

    # Wpływ spadku ciśnienia chłodziwa na temperaturę narzędzia
    pressure_drop_flag = coolant_press < 18.5
    temp_tool = np.where(pressure_drop_flag, temp_tool + np.random.normal(30, 10), temp_tool)

    # ZMIENNE DOCELOWE
    # Chropowatość Ra oparta na modelu kinematycznym (f^2 / 8r) i wibracjach
    # R_epsilon (promień zaokrąglenia płytki) przyjmujemy jako 0.8 mm
    ra_kinematic = (f**2) / (8 * 0.8) * 1000 # *1000 dla konwersji jednostek
    ra_actual = ra_kinematic + 0.8 * vb_wear + 0.15 * vibrations + np.random.normal(0, 0.1, n_samples)

    # Odchylenie wymiarowe (skutek odpychania z powodu zużycia + rozszerzalność cieplna)
    dimension_dev = 0.005 + 0.05 * vb_wear + 0.0001 * (temp_tool - 100) + np.random.normal(0, 0.002, n_samples)

    # W przypadku pęknięcia narzędzia parametry jakościowe ulegają destrukcji
    ra_actual = np.where(tool_break_flag == 1, ra_actual + np.random.uniform(3.0, 5.0), ra_actual)
    dimension_dev = np.where(tool_break_flag == 1, dimension_dev + np.random.uniform(0.1, 0.3), dimension_dev)

    # Klasyfikacja jakości na podstawie rygorystycznych limitów przemysłowych
    # OK (Ra <= 1.6 i dev <= 0.03)
    # Do_Poprawy (Ra <= 3.2 i dev <= 0.05)
    # Zlom (Ra > 3.2 lub dev > 0.05)
    conditions = [
        (ra_actual > 3.2) | (np.abs(dimension_dev) > 0.05),
        (ra_actual > 1.6) | (np.abs(dimension_dev) > 0.03)
    ]
    choices = ['Zlom', 'Do_Poprawy']
    quality_class = np.select(conditions, choices, default='OK')

    # TWORZENIE DATAFRAME ZE WSZYSTKIMI PARAMETRAMI
    df = pd.DataFrame({
        'ID_Czesci': np.arange(1, n_samples + 1),
        'ID_Narzedzia': tool_id,
        'Predkosc_Skrawania_vc': np.round(vc, 2),
        'Posuw_f': np.round(f, 4),
        'Glebokosc_Skrawania_ap': np.round(ap, 2),
        'Twardosc_Odlewu_HB': np.round(hb, 1),
        'Zuzycie_Narzedzia_VB': np.round(vb_wear, 3),
        'Temp_Narzedzia_Max_C': np.round(temp_tool, 1),
        'Temp_Chlodziwa_Mean_C': np.round(coolant_temp, 1),
        'Cisnienie_Chlodziwa_Min_bar': np.round(coolant_press, 2),
        'Wibracje_Wrzeciona_RMS_g': np.round(vibrations, 3),
        'Pobor_Mocy_Mean_kW': np.round(power_kw, 2),
        'Awaria_Narzedzia': tool_break_flag,
        'Chropowatosc_Ra': np.round(ra_actual, 3),
        'Odchylenie_Wymiarowe': np.round(dimension_dev, 4),
        'Klasa_Jakosci': quality_class
    })

    return df

# Wywołanie funkcji i zapis danych do pliku
df_cnc = generate_machining_dataset(30000)
df_cnc.to_csv('dane_obrobka_cnc.csv', index=False)
print("Dane wygenerowane! Wygenerowano: ", len(df_cnc), "wierszy. Plik zapisano jako 'dane_obrobka_cnc.csv'.")

Rozpoczęto generowanie zbioru danych...
Dane wygenerowane! Wygenerowano:  30000 wierszy. Plik zapisano jako 'dane_obrobka_cnc.csv'.
